# Анализ клиентского поведения и монетизации тарифов мобильного оператора

**Формат проекта:** аналитическое исследование в Jupyter Notebook.

## Бизнес-контекст

Мобильный оператор предлагает два тарифных плана — **Smart** и **Ultra**.  
Задача анализа — понять, как различаются потребительское поведение и выручка по тарифам, а также определить, какие факторы следует учитывать при выборе тарифа для маркетингового продвижения.

## Цель исследования

Сформировать аналитическую витрину уровня **«пользователь — месяц»**, рассчитать фактическую выручку оператора и ответить на четыре вопроса:

1. Как различается использование звонков, сообщений и мобильного интернета между тарифами?
2. Какая часть выручки формируется абонентской платой, а какая — сверхлимитным потреблением?
3. Связан ли тарифный план со средней месячной выручкой на пользователя?
4. Отличается ли средняя месячная выручка клиентов Москвы от выручки клиентов других регионов?

## Результаты проекта

В ноутбуке последовательно выполняются:

- загрузка и проверка качества данных;
- подготовка журналов использования услуг;
- построение полного календаря активных клиентских месяцев;
- расчёт сверхлимитного потребления и месячной выручки;
- анализ ключевых продуктовых и коммерческих метрик;
- статистическая проверка различий;
- формирование управленческого резюме и экспорт результатов.

## Методологические допущения

- Нулевая длительность звонка интерпретируется как пропущенный вызов и сохраняется в данных.
- Продолжительность каждого звонка округляется вверх до полной минуты в соответствии с правилами тарификации.
- Сверхлимитный интернет округляется вверх до полного гигабайта.
- Абонентская плата начисляется за каждый месяц, в котором клиент числился активным, даже при отсутствии событий в журналах использования.
- Статистические тесты выполняются на уровне **пользователя**, а не отдельных пользовательских месяцев. Это снижает риск искусственного завышения объёма выборки из-за повторных наблюдений одного и того же клиента.
- Наблюдаемые различия являются ассоциациями. Без экспериментального дизайна анализ не доказывает, что именно тариф или регион причинно изменяют выручку.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from scipy import stats as st

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

sns.set_theme(context="notebook")
ALPHA = 0.05

## 1. Параметры проекта

In [ ]:
# Измените только эту ячейку, если файлы находятся в другом каталоге.
DATA_DIR = Path("/datasets")
OUTPUT_DIR = Path("outputs")

# При False результаты будут показаны в ноутбуке, но не записаны на диск.
SAVE_RESULTS = True

REQUIRED_FILES = {
    "calls": "calls.csv",
    "sessions": "internet.csv",
    "messages": "messages.csv",
    "tariffs": "tariffs.csv",
    "users": "users.csv",
}

REQUIRED_COLUMNS = {
    "calls": {"user_id", "call_date", "duration"},
    "sessions": {"user_id", "session_date", "mb_used"},
    "messages": {"user_id", "message_date"},
    "tariffs": {
        "tariff_name",
        "rub_monthly_fee",
        "minutes_included",
        "messages_included",
        "mb_per_month_included",
        "rub_per_minute",
        "rub_per_message",
        "rub_per_gb",
    },
    "users": {"user_id", "city", "tariff", "reg_date", "churn_date"},
}

## 2. Загрузка и первичный аудит данных

In [ ]:
def load_data(data_dir: Path) -> dict[str, pd.DataFrame]:
    """Загружает исходные таблицы и проверяет обязательную структуру."""
    datasets = {}

    for dataset_name, filename in REQUIRED_FILES.items():
        file_path = data_dir / filename

        if not file_path.exists():
            raise FileNotFoundError(
                f"Не найден файл: {file_path}. "
                "Проверьте значение DATA_DIR в предыдущей ячейке."
            )

        frame = pd.read_csv(file_path)
        missing_columns = REQUIRED_COLUMNS[dataset_name] - set(frame.columns)

        if missing_columns:
            raise ValueError(
                f"В таблице {filename} отсутствуют поля: "
                f"{', '.join(sorted(missing_columns))}"
            )

        datasets[dataset_name] = frame

    return datasets


datasets = load_data(DATA_DIR)

calls = datasets["calls"]
sessions = datasets["sessions"]
messages = datasets["messages"]
tariffs = datasets["tariffs"]
users = datasets["users"]

print("Исходные таблицы успешно загружены.")

In [ ]:
def build_data_audit(datasets: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Формирует компактный отчёт о размере и качестве исходных таблиц."""
    audit_rows = []

    for name, frame in datasets.items():
        audit_rows.append(
            {
                "table": name,
                "rows": len(frame),
                "columns": frame.shape[1],
                "duplicate_rows": int(frame.duplicated().sum()),
                "missing_cells": int(frame.isna().sum().sum()),
                "missing_share": frame.isna().mean().mean(),
            }
        )

    return pd.DataFrame(audit_rows)


data_audit = build_data_audit(datasets)
display(data_audit.style.format({"missing_share": "{:.2%}"}))

### Состав данных

| Таблица | Единица наблюдения | Основное содержание |
|---|---|---|
| `users` | клиент | город, тариф, даты регистрации и расторжения |
| `calls` | звонок | дата и продолжительность |
| `messages` | сообщение | дата отправки |
| `internet` | интернет-сессия | дата и объём трафика |
| `tariffs` | тариф | лимиты, абонентская плата и стоимость сверхлимитных услуг |

Пропуски в `churn_date` являются ожидаемыми: отсутствие даты означает, что клиент не расторг договор в пределах периода наблюдения.

In [ ]:
# Просмотр примеров записей без вывода больших массивов данных.
for name, frame in datasets.items():
    display(Markdown(f"#### `{name}`"))
    display(frame.head())

## 3. Подготовка данных

In [ ]:
data = {name: frame.copy() for name, frame in datasets.items()}

# Приведение дат.
data["users"]["reg_date"] = pd.to_datetime(
    data["users"]["reg_date"], errors="raise"
)
data["users"]["churn_date"] = pd.to_datetime(
    data["users"]["churn_date"], errors="coerce"
)
data["calls"]["call_date"] = pd.to_datetime(
    data["calls"]["call_date"], errors="raise"
)
data["messages"]["message_date"] = pd.to_datetime(
    data["messages"]["message_date"], errors="raise"
)
data["sessions"]["session_date"] = pd.to_datetime(
    data["sessions"]["session_date"], errors="raise"
)

# Правила тарификации и техническая очистка.
data["calls"]["duration"] = np.ceil(
    data["calls"]["duration"].clip(lower=0)
).astype("int64")

data["sessions"]["mb_used"] = data["sessions"]["mb_used"].clip(lower=0)
data["sessions"] = data["sessions"].drop(
    columns=["Unnamed: 0"], errors="ignore"
)

data["tariffs"] = data["tariffs"].rename(
    columns={"tariff_name": "tariff"}
)

# Месяц хранится как Period[M]: это исключает смешивание одинаковых
# номеров месяцев из разных лет.
for dataset_name, date_column in {
    "calls": "call_date",
    "messages": "message_date",
    "sessions": "session_date",
}.items():
    data[dataset_name]["month"] = (
        data[dataset_name][date_column].dt.to_period("M")
    )

print("Подготовка типов и применение правил тарификации завершены.")

In [ ]:
quality_checks = pd.Series(
    {
        "negative_call_duration": int((data["calls"]["duration"] < 0).sum()),
        "negative_internet_usage": int((data["sessions"]["mb_used"] < 0).sum()),
        "duplicated_user_ids": int(data["users"]["user_id"].duplicated().sum()),
        "unknown_user_tariffs": int(
            (~data["users"]["tariff"].isin(data["tariffs"]["tariff"])).sum()
        ),
    },
    name="count",
).to_frame()

display(quality_checks)

## 4. Построение витрины «пользователь — месяц»

Простое объединение агрегатов звонков, сообщений и интернет-сессий учитывает только месяцы, в которых клиент совершал хотя бы одно действие. Такой подход занижает абонентскую выручку: клиент мог не пользоваться услугами, но оставаться активным и оплачивать тариф.

Поэтому сначала формируется полный календарь активных месяцев каждого клиента — от месяца регистрации до месяца расторжения либо до конца периода наблюдения.

In [ ]:
def get_observation_end(data: dict[str, pd.DataFrame]) -> pd.Timestamp:
    """Возвращает последнюю дату, представленную в журналах использования."""
    latest_dates = [
        data["calls"]["call_date"].max(),
        data["messages"]["message_date"].max(),
        data["sessions"]["session_date"].max(),
    ]
    return max(latest_dates)


def build_active_user_months(
    users: pd.DataFrame,
    observation_end: pd.Timestamp,
) -> pd.DataFrame:
    """Формирует полный календарь активных месяцев для каждого клиента."""
    records = []
    observation_end_period = observation_end.to_period("M")

    for user in users.itertuples(index=False):
        start_month = user.reg_date.to_period("M")
        end_date = (
            user.churn_date
            if pd.notna(user.churn_date)
            else observation_end
        )
        end_month = min(end_date.to_period("M"), observation_end_period)

        if end_month < start_month:
            continue

        for month in pd.period_range(
            start=start_month,
            end=end_month,
            freq="M",
        ):
            records.append(
                {"user_id": user.user_id, "month": month}
            )

    return pd.DataFrame(records)


observation_end = get_observation_end(data)
active_user_months = build_active_user_months(
    data["users"],
    observation_end,
)

print(f"Конец периода наблюдения: {observation_end.date()}")
print(f"Сформировано клиентских месяцев: {len(active_user_months):,}")

In [ ]:
calls_monthly = (
    data["calls"]
    .groupby(["user_id", "month"], as_index=False)
    .agg(
        calls=("duration", "size"),
        minutes=("duration", "sum"),
    )
)

messages_monthly = (
    data["messages"]
    .groupby(["user_id", "month"], as_index=False)
    .agg(messages=("message_date", "size"))
)

sessions_monthly = (
    data["sessions"]
    .groupby(["user_id", "month"], as_index=False)
    .agg(mb_used=("mb_used", "sum"))
)

monthly_usage = (
    calls_monthly
    .merge(
        messages_monthly,
        on=["user_id", "month"],
        how="outer",
    )
    .merge(
        sessions_monthly,
        on=["user_id", "month"],
        how="outer",
    )
)

display(monthly_usage.head())

In [ ]:
user_attributes = data["users"].drop_duplicates(subset="user_id")

user_month = (
    active_user_months
    .merge(
        monthly_usage,
        on=["user_id", "month"],
        how="left",
    )
    .merge(
        user_attributes,
        on="user_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        data["tariffs"],
        on="tariff",
        how="left",
        validate="many_to_one",
    )
)

usage_columns = ["calls", "minutes", "messages", "mb_used"]
user_month[usage_columns] = user_month[usage_columns].fillna(0)
user_month[["calls", "minutes", "messages"]] = user_month[
    ["calls", "minutes", "messages"]
].astype("int64")

if user_month["rub_monthly_fee"].isna().any():
    unknown_tariffs = sorted(
        user_month.loc[
            user_month["rub_monthly_fee"].isna(),
            "tariff",
        ]
        .dropna()
        .unique()
    )
    raise ValueError(
        f"Не найдены параметры тарифов: {unknown_tariffs}"
    )

display(user_month.head())
print(f"Размер аналитической витрины: {user_month.shape}")

## 5. Расчёт сверхлимитного потребления и выручки

Для каждого клиентского месяца рассчитываются:

- превышение включённого лимита минут;
- превышение лимита сообщений;
- превышение лимита интернет-трафика;
- выручка от каждой сверхлимитной услуги;
- общая месячная выручка как сумма абонентской платы и сверхлимитных начислений.

In [ ]:
user_month["paid_minutes"] = (
    user_month["minutes"] - user_month["minutes_included"]
).clip(lower=0)

user_month["paid_messages"] = (
    user_month["messages"] - user_month["messages_included"]
).clip(lower=0)

user_month["paid_mb"] = (
    user_month["mb_used"]
    - user_month["mb_per_month_included"]
).clip(lower=0)

user_month["paid_gb"] = np.ceil(
    user_month["paid_mb"] / 1024
).astype("int64")

user_month["revenue_minutes"] = (
    user_month["paid_minutes"]
    * user_month["rub_per_minute"]
)
user_month["revenue_messages"] = (
    user_month["paid_messages"]
    * user_month["rub_per_message"]
)
user_month["revenue_internet"] = (
    user_month["paid_gb"]
    * user_month["rub_per_gb"]
)

user_month["overage_revenue"] = user_month[
    [
        "revenue_minutes",
        "revenue_messages",
        "revenue_internet",
    ]
].sum(axis=1)

user_month["monthly_revenue"] = (
    user_month["rub_monthly_fee"]
    + user_month["overage_revenue"]
)

user_month["gb_used"] = user_month["mb_used"] / 1024
user_month["month_start"] = user_month["month"].dt.to_timestamp()

display(
    user_month[
        [
            "user_id",
            "month",
            "tariff",
            "minutes",
            "messages",
            "gb_used",
            "overage_revenue",
            "monthly_revenue",
        ]
    ].head()
)

In [ ]:
revenue_checks = pd.Series(
    {
        "negative_monthly_revenue": int(
            (user_month["monthly_revenue"] < 0).sum()
        ),
        "revenue_below_monthly_fee": int(
            (
                user_month["monthly_revenue"]
                < user_month["rub_monthly_fee"]
            ).sum()
        ),
        "missing_revenue": int(
            user_month["monthly_revenue"].isna().sum()
        ),
    },
    name="count",
).to_frame()

display(revenue_checks)

## 6. Ключевые продуктовые и коммерческие показатели

In [ ]:
monthly_stats = (
    user_month
    .groupby(["month_start", "tariff"], as_index=False)
    .agg(
        active_users=("user_id", "nunique"),
        calls_mean=("calls", "mean"),
        minutes_mean=("minutes", "mean"),
        messages_mean=("messages", "mean"),
        gb_mean=("gb_used", "mean"),
        revenue_mean=("monthly_revenue", "mean"),
        revenue_median=("monthly_revenue", "median"),
        revenue_std=("monthly_revenue", "std"),
    )
    .round(2)
)

tariff_kpis = (
    user_month
    .groupby("tariff", as_index=False)
    .agg(
        subscribers=("user_id", "nunique"),
        user_months=("user_id", "size"),
        avg_monthly_revenue=("monthly_revenue", "mean"),
        median_monthly_revenue=("monthly_revenue", "median"),
        avg_minutes=("minutes", "mean"),
        avg_messages=("messages", "mean"),
        avg_gb=("gb_used", "mean"),
        total_revenue=("monthly_revenue", "sum"),
        total_overage_revenue=("overage_revenue", "sum"),
    )
)

tariff_kpis["overage_revenue_share"] = (
    tariff_kpis["total_overage_revenue"]
    / tariff_kpis["total_revenue"]
)

tariff_kpis = tariff_kpis.round(2)

display(Markdown("### Сводные KPI по тарифам"))
display(
    tariff_kpis.style.format(
        {
            "avg_monthly_revenue": "{:,.2f}",
            "median_monthly_revenue": "{:,.2f}",
            "avg_minutes": "{:,.1f}",
            "avg_messages": "{:,.1f}",
            "avg_gb": "{:,.1f}",
            "total_revenue": "{:,.0f}",
            "total_overage_revenue": "{:,.0f}",
            "overage_revenue_share": "{:.1%}",
        }
    )
)

In [ ]:
churn_rate = data["users"]["churn_date"].notna().mean()

portfolio_summary = pd.Series(
    {
        "unique_clients": data["users"]["user_id"].nunique(),
        "client_months": len(user_month),
        "observation_start": user_month["month_start"].min().date(),
        "observation_end": user_month["month_start"].max().date(),
        "clients_with_churn_date": churn_rate,
    },
    name="value",
).to_frame()

display(portfolio_summary)

## 7. Визуальный анализ динамики

In [ ]:
def plot_monthly_metric(
    metric: str,
    title: str,
    ylabel: str,
    filename: str,
) -> None:
    """Показывает и при необходимости сохраняет месячную динамику KPI."""
    plt.figure(figsize=(11, 6))
    sns.lineplot(
        data=monthly_stats,
        x="month_start",
        y=metric,
        hue="tariff",
        marker="o",
    )
    plt.title(title)
    plt.xlabel("Месяц")
    plt.ylabel(ylabel)
    plt.xticks(rotation=45)
    plt.tight_layout()

    if SAVE_RESULTS:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        plt.savefig(OUTPUT_DIR / filename, dpi=160)

    plt.show()

In [ ]:
plot_monthly_metric(
    metric="minutes_mean",
    title="Среднее использование голосовой связи по тарифам",
    ylabel="Минуты на активного клиента",
    filename="monthly_minutes.png",
)

In [ ]:
plot_monthly_metric(
    metric="messages_mean",
    title="Среднее количество сообщений по тарифам",
    ylabel="Сообщения на активного клиента",
    filename="monthly_messages.png",
)

In [ ]:
plot_monthly_metric(
    metric="gb_mean",
    title="Среднее потребление мобильного интернета по тарифам",
    ylabel="ГБ на активного клиента",
    filename="monthly_internet.png",
)

In [ ]:
plot_monthly_metric(
    metric="revenue_mean",
    title="Средняя месячная выручка по тарифам",
    ylabel="Выручка на активного клиента, руб.",
    filename="monthly_revenue.png",
)

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(
    data=user_month,
    x="monthly_revenue",
    hue="tariff",
    bins=40,
    element="step",
    stat="density",
    common_norm=False,
)
plt.title("Распределение месячной выручки по тарифам")
plt.xlabel("Месячная выручка, руб.")
plt.ylabel("Плотность распределения")
plt.tight_layout()

if SAVE_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    plt.savefig(
        OUTPUT_DIR / "revenue_distribution.png",
        dpi=160,
    )

plt.show()

In [ ]:
revenue_structure = (
    user_month
    .groupby("tariff", as_index=False)
    .agg(
        subscription_revenue=("rub_monthly_fee", "sum"),
        overage_revenue=("overage_revenue", "sum"),
    )
    .set_index("tariff")
)

revenue_structure.plot(
    kind="bar",
    stacked=True,
    figsize=(9, 6),
)
plt.title("Структура выручки по тарифам")
plt.xlabel("Тариф")
plt.ylabel("Выручка, руб.")
plt.xticks(rotation=0)
plt.legend(["Абонентская плата", "Сверхлимитная выручка"])
plt.tight_layout()

if SAVE_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    plt.savefig(
        OUTPUT_DIR / "revenue_structure.png",
        dpi=160,
    )

plt.show()

## 8. Проверка статистических гипотез

Для сравнения групп применяется двусторонний **t-тест Уэлча**. Он не требует равенства дисперсий и лучше подходит для групп неодинакового размера.

Единицей анализа является клиент. Для каждого клиента сначала рассчитывается его средняя месячная выручка за доступный период, после чего сравниваются независимые группы.

Дополнительно рассчитываются:

- разность средних;
- 95%-й доверительный интервал;
- стандартизированный размер эффекта Cohen's d;
- статистическая значимость при уровне `α = 0,05`.

In [ ]:
user_level_revenue = (
    user_month
    .groupby(
        ["user_id", "tariff", "city"],
        as_index=False,
    )
    .agg(
        avg_monthly_revenue=("monthly_revenue", "mean"),
        observed_months=("month", "nunique"),
    )
)

display(user_level_revenue.head())

In [ ]:
def welch_test(
    sample_a: pd.Series,
    sample_b: pd.Series,
    comparison: str,
    group_a: str,
    group_b: str,
    alpha: float = ALPHA,
) -> dict:
    """Выполняет тест Уэлча, строит доверительный интервал и Cohen's d."""
    a = sample_a.dropna().astype(float)
    b = sample_b.dropna().astype(float)

    if len(a) < 2 or len(b) < 2:
        raise ValueError(
            f"Недостаточно наблюдений для сравнения: {comparison}"
        )

    test_result = st.ttest_ind(a, b, equal_var=False)

    mean_diff = a.mean() - b.mean()
    var_a = a.var(ddof=1)
    var_b = b.var(ddof=1)

    variance_term = var_a / len(a) + var_b / len(b)
    standard_error = np.sqrt(variance_term)

    welch_df = variance_term**2 / (
        (var_a / len(a)) ** 2 / (len(a) - 1)
        + (var_b / len(b)) ** 2 / (len(b) - 1)
    )

    critical_value = st.t.ppf(
        1 - alpha / 2,
        welch_df,
    )

    pooled_sd = np.sqrt(
        (
            (len(a) - 1) * var_a
            + (len(b) - 1) * var_b
        )
        / (len(a) + len(b) - 2)
    )

    cohens_d = (
        mean_diff / pooled_sd
        if pooled_sd > 0
        else np.nan
    )

    return {
        "comparison": comparison,
        "group_a": group_a,
        "group_b": group_b,
        "n_a": len(a),
        "n_b": len(b),
        "mean_a": a.mean(),
        "mean_b": b.mean(),
        "mean_difference": mean_diff,
        "ci_95_low": mean_diff - critical_value * standard_error,
        "ci_95_high": mean_diff + critical_value * standard_error,
        "p_value": float(test_result.pvalue),
        "cohens_d": float(cohens_d),
        "significant": bool(test_result.pvalue < alpha),
    }

### Гипотеза 1. Средняя месячная выручка различается между тарифами

- **H₀:** средняя месячная выручка клиентов Ultra и Smart одинакова.
- **H₁:** средняя месячная выручка клиентов Ultra и Smart различается.

In [ ]:
tariff_test = welch_test(
    user_level_revenue.loc[
        user_level_revenue["tariff"] == "ultra",
        "avg_monthly_revenue",
    ],
    user_level_revenue.loc[
        user_level_revenue["tariff"] == "smart",
        "avg_monthly_revenue",
    ],
    comparison="Ultra vs Smart",
    group_a="Ultra",
    group_b="Smart",
)

display(
    pd.DataFrame([tariff_test]).style.format(
        {
            "mean_a": "{:,.2f}",
            "mean_b": "{:,.2f}",
            "mean_difference": "{:,.2f}",
            "ci_95_low": "{:,.2f}",
            "ci_95_high": "{:,.2f}",
            "p_value": "{:.6f}",
            "cohens_d": "{:.3f}",
        }
    )
)

### Гипотеза 2. Средняя месячная выручка клиентов Москвы отличается от других регионов

- **H₀:** средняя месячная выручка клиентов Москвы и других регионов одинакова.
- **H₁:** средняя месячная выручка клиентов Москвы и других регионов различается.

In [ ]:
region_test = welch_test(
    user_level_revenue.loc[
        user_level_revenue["city"] == "Москва",
        "avg_monthly_revenue",
    ],
    user_level_revenue.loc[
        user_level_revenue["city"] != "Москва",
        "avg_monthly_revenue",
    ],
    comparison="Москва vs другие регионы",
    group_a="Москва",
    group_b="Другие регионы",
)

hypothesis_tests = pd.DataFrame(
    [tariff_test, region_test]
)

display(
    hypothesis_tests.style.format(
        {
            "mean_a": "{:,.2f}",
            "mean_b": "{:,.2f}",
            "mean_difference": "{:,.2f}",
            "ci_95_low": "{:,.2f}",
            "ci_95_high": "{:,.2f}",
            "p_value": "{:.6f}",
            "cohens_d": "{:.3f}",
        }
    )
)

In [ ]:
def effect_size_label(value: float) -> str:
    """Возвращает содержательную интерпретацию абсолютного Cohen's d."""
    effect = abs(value)

    if effect < 0.2:
        return "пренебрежимо малый"
    if effect < 0.5:
        return "малый"
    if effect < 0.8:
        return "средний"
    return "крупный"


def interpret_test(result: dict) -> str:
    """Формирует деловую интерпретацию статистического теста."""
    direction = (
        f"выше в группе «{result['group_a']}»"
        if result["mean_difference"] > 0
        else f"выше в группе «{result['group_b']}»"
    )

    significance = (
        "Различие статистически значимо"
        if result["significant"]
        else "Статистически значимое различие не обнаружено"
    )

    return (
        f"**{result['comparison']}.** {significance} "
        f"(p = {result['p_value']:.4f}). "
        f"Средняя выручка {direction}; абсолютная разность составляет "
        f"{abs(result['mean_difference']):,.2f} руб. "
        f"95%-й доверительный интервал разности: "
        f"[{result['ci_95_low']:,.2f}; {result['ci_95_high']:,.2f}] руб. "
        f"Размер эффекта — {effect_size_label(result['cohens_d'])} "
        f"(Cohen's d = {result['cohens_d']:.3f})."
    )


display(Markdown(interpret_test(tariff_test)))
display(Markdown(interpret_test(region_test)))

## 9. Управленческое резюме

In [ ]:
def build_executive_summary(
    tariff_kpis: pd.DataFrame,
    tariff_test: dict,
    region_test: dict,
) -> str:
    """Создаёт краткое резюме, основанное на рассчитанных показателях."""
    kpis = tariff_kpis.set_index("tariff")

    revenue_leader = kpis["avg_monthly_revenue"].idxmax()
    usage_leader = kpis["avg_gb"].idxmax()
    overage_leader = kpis["overage_revenue_share"].idxmax()

    tariff_conclusion = (
        "различие подтверждается статистически"
        if tariff_test["significant"]
        else "данных недостаточно, чтобы подтвердить устойчивое различие"
    )

    region_conclusion = (
        "региональные группы различаются статистически"
        if region_test["significant"]
        else "устойчивого различия между Москвой и другими регионами не обнаружено"
    )

    return f"""
### Основные результаты

1. Наибольшая средняя месячная выручка наблюдается у тарифа **{revenue_leader}**.
2. Наиболее интенсивное среднее потребление интернет-трафика характерно для тарифа **{usage_leader}**.
3. Наибольшая зависимость выручки от сверхлимитных начислений отмечена у тарифа **{overage_leader}**.
4. При сравнении тарифов {tariff_conclusion}.
5. При сравнении клиентов по региону {region_conclusion}.

### Интерпретация для бизнеса

Более высокая выручка сама по себе не означает, что тариф выгоднее для продвижения. Для итогового решения необходимо сопоставить выручку с:

- себестоимостью голосовой связи и интернет-трафика;
- маржинальностью тарифов;
- стоимостью привлечения клиента;
- оттоком и сроком жизни клиента;
- частотой превышения лимитов и риском негативного клиентского опыта;
- вероятностью перехода между тарифами.

Текущий анализ отвечает на вопрос о поведении и выручке, но не заменяет расчёт **LTV, CAC и contribution margin**.
"""


display(
    Markdown(
        build_executive_summary(
            tariff_kpis,
            tariff_test,
            region_test,
        )
    )
)

## 10. Ограничения исследования

1. Данные являются наблюдательными, поэтому выявленные связи нельзя интерпретировать как причинные эффекты тарифа.
2. В выборке отсутствуют прямые данные о затратах, марже, привлечении и удержании клиентов.
3. Средняя выручка чувствительна к выбросам; поэтому вместе со средним анализируется медиана и распределение.
4. Для более строгой оценки динамики можно использовать панельную регрессию или модель со случайными эффектами.
5. Для продуктового решения полезно дополнить анализ сегментацией клиентов, оттоком, переходами между тарифами и прогнозом LTV.

## 11. Экспорт аналитических результатов

In [ ]:
if SAVE_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    user_month.to_csv(
        OUTPUT_DIR / "user_month_behavior.csv",
        index=False,
    )
    monthly_stats.to_csv(
        OUTPUT_DIR / "monthly_tariff_stats.csv",
        index=False,
    )
    tariff_kpis.to_csv(
        OUTPUT_DIR / "tariff_kpis.csv",
        index=False,
    )
    user_level_revenue.to_csv(
        OUTPUT_DIR / "user_level_revenue.csv",
        index=False,
    )
    hypothesis_tests.to_csv(
        OUTPUT_DIR / "hypothesis_tests.csv",
        index=False,
    )

    print(
        f"Таблицы и графики сохранены в: "
        f"{OUTPUT_DIR.resolve()}"
    )
else:
    print("Экспорт отключён: SAVE_RESULTS = False")

---

### Итог

Ноутбук представляет полный воспроизводимый аналитический цикл: от контроля исходных данных до расчёта бизнес-метрик, статистической проверки и интерпретации результатов. Вся исполняемая логика находится непосредственно в Jupyter Notebook; отдельный Python-скрипт для запуска проекта не требуется.